In [ ]:
#imports
import numpy as np
import pandas as pd
import geopandas as gpd
import holidays
from pygam import LinearGAM, s, f
from datetime import timedelta
from pathlib import Path
from tqdm import tqdm  
from rapidfuzz import process, fuzz
import os 
from quantile_forest import RandomForestQuantileRegressor
import pyarrow.parquet as pq
import pyarrow.fs as pafs
from config import (BA_COL, TARGET, BA_TO_INTERCONNECT, AGGREGATE_NODES, INTERCONNECTS,
                    TRAIN_START, TRAIN_END, TEST_START, TEST_END,
                    RESULTS_DIR, KEPT_BAS_PATH,
                    select_kept_bas, save_kept_bas, members, add_hourly_lags)

In [ ]:
#formatting
temp_wide = pd.read_parquet("/home/s13q981/HTL/ba_hourly_temp.parquet")
df_temp = (
    temp_wide.reset_index()
    .drop(columns="XTIME")
    .melt(id_vars="Time", var_name="balancing_authority_code_eia", value_name="avg_temp_K")
    .rename(columns={"Time": "datetime_utc"})
    .sort_values(["balancing_authority_code_eia", "datetime_utc"])
    .reset_index(drop=True)
)

In [ ]:
#smoothed temperature variables
alpha_95, alpha_99 = 0.05, 0.01
df_temp["temp_s95"] = df_temp.groupby("balancing_authority_code_eia")["avg_temp_K"].transform(
    lambda x: x.ewm(alpha=alpha_95, adjust=False).mean()
)
df_temp["temp_s99"] = df_temp.groupby("balancing_authority_code_eia")["avg_temp_K"].transform(
    lambda x: x.ewm(alpha=alpha_99, adjust=False).mean()
)

In [ ]:
# continuity check:
print(df_temp.groupby("balancing_authority_code_eia")["datetime_utc"].agg(["min", "max", "count"]))
gaps = df_temp.groupby("balancing_authority_code_eia")["datetime_utc"].diff().value_counts()
print(gaps)

In [ ]:
#load EIA 930 via PUDL
s3 = pafs.S3FileSystem(anonymous=True, region="us-west-2")
df_y = pq.read_table(
    "pudl.catalyst.coop/nightly/core_eia930__hourly_operations.parquet",
    filesystem=s3
).to_pandas(types_mapper=pd.ArrowDtype)

df_y["datetime_utc"] = df_y["datetime_utc"].astype("datetime64[ns]")  # match df_temp's dtype for the merge key

print(df_y.shape)
print(df_y.columns.tolist())

In [ ]:
#merge with EIA
df_merged = df_y.merge(
    df_temp,   # df_temp carries temp_s95/temp_s99
    on=[BA_COL, "datetime_utc"],
    how="inner",
)
df_merged[BA_COL] = df_merged[BA_COL].astype(str)
df_merged = df_merged.sort_values([BA_COL, "datetime_utc"]).reset_index(drop=True)

kept_bas, coverage = select_kept_bas(df_merged)
os.makedirs(RESULTS_DIR, exist_ok=True)
save_kept_bas(kept_bas, coverage)
dropped = coverage[~coverage.index.isin(kept_bas)]
print(f"Kept {len(kept_bas)} / {len(BA_TO_INTERCONNECT)} mapped BAs; dropped (test hours / max {coverage.max()}):")
print(dropped.to_string())
df_merged = df_merged[df_merged[BA_COL].isin(kept_bas)].reset_index(drop=True)

# lags by timestamp, not by row (row shifts pull the wrong hour across gaps)
df_merged = add_hourly_lags(df_merged)

print(df_merged.shape)
print(df_merged[BA_COL].nunique(), "BAs after merge + coverage filter")
print(df_merged[["datetime_utc", BA_COL, TARGET, "avg_temp_K", "load_24", "load_168"]].head())

In [ ]:
#calender variables
df_merged["hour"]      = df_merged["datetime_utc"].dt.hour
df_merged["dayofweek"] = df_merged["datetime_utc"].dt.dayofweek
df_merged["dayofyear"] = df_merged["datetime_utc"].dt.dayofyear
us_holidays = holidays.US()
df_merged["is_holiday"] = df_merged["datetime_utc"].dt.date.apply(lambda x: x in us_holidays).astype(int)

print(df_merged.shape)
print(df_merged["balancing_authority_code_eia"].nunique(), "BAs after merge")
print(df_merged[["datetime_utc","balancing_authority_code_eia","demand_adjusted_mwh",
                  "avg_temp_K","temp_s95","temp_s99","load_24","load_168","hour","is_holiday"]].head())


In [ ]:
#interconnect / national aggregation
# Builds EASTERN / WESTERN / ERCOT / NATIONAL as pseudo-BA rows so the GAM and XGB cells
# also produce the independent top-level base forecasts MinT needs.

present_bas = set(df_merged[BA_COL].unique())
assert present_bas == set(kept_bas), "df_merged should contain exactly the kept BAs at this point"

train_rows = (df_merged["datetime_utc"] >= TRAIN_START) & (df_merged["datetime_utc"] <= TRAIN_END)
ba_weights = df_merged.loc[train_rows].groupby(BA_COL)[TARGET].mean()

dem_wide = df_merged.pivot(index="datetime_utc", columns=BA_COL, values=TARGET)

temp_wide = df_temp.pivot(index="datetime_utc", columns=BA_COL, values="avg_temp_K")

def build_aggregate(node):
    mem = members(node, kept_bas)
    w = ba_weights.reindex(mem)
    assert w.notna().all(), f"{node}: missing training-period weight for {w[w.isna()].index.tolist()}"
    w = w / w.sum()

    t = temp_wide[mem].dropna()
    agg_t = pd.DataFrame({"avg_temp_K": t.to_numpy() @ w.to_numpy()}, index=t.index)
    agg_t["temp_s95"] = agg_t["avg_temp_K"].ewm(alpha=alpha_95, adjust=False).mean()
    agg_t["temp_s99"] = agg_t["avg_temp_K"].ewm(alpha=alpha_99, adjust=False).mean()

    d = dem_wide.reindex(columns=mem)
    complete = d.notna().all(axis=1)
    agg_d = d[complete].sum(axis=1).rename(TARGET)

    out = agg_t.join(agg_d, how="inner").reset_index()
    out[BA_COL] = node
    print(f"{node}: {len(mem)} members, {complete.sum()} complete hours "
          f"({(~complete).sum()} hours dropped because a member was missing)")
    return out

df_agg = pd.concat([build_aggregate(n) for n in AGGREGATE_NODES], ignore_index=True)
df_agg = df_agg.sort_values([BA_COL, "datetime_utc"]).reset_index(drop=True)

# calendar variables, same logic as the BA-level cell above
df_agg["hour"]      = df_agg["datetime_utc"].dt.hour
df_agg["dayofweek"] = df_agg["datetime_utc"].dt.dayofweek
df_agg["dayofyear"] = df_agg["datetime_utc"].dt.dayofyear
df_agg["is_holiday"] = df_agg["datetime_utc"].dt.date.apply(lambda x: x in us_holidays).astype(int)

df_agg = add_hourly_lags(df_agg)

agg_cols = ["datetime_utc", BA_COL, TARGET,
            "avg_temp_K", "temp_s95", "temp_s99", "load_24", "load_168",
            "hour", "dayofweek", "dayofyear", "is_holiday"]
df_merged = pd.concat([df_merged[agg_cols], df_agg[agg_cols]], ignore_index=True)
df_merged = df_merged.sort_values([BA_COL, "datetime_utc"]).reset_index(drop=True)

# sanity check: every aggregate hour equals the sum of its members
chk = df_merged.pivot(index="datetime_utc", columns=BA_COL, values=TARGET)
for node in AGGREGATE_NODES:
    rows = chk[node].dropna()
    gap = (rows - chk.loc[rows.index, members(node, kept_bas)].sum(axis=1)).abs().max()
    assert gap < 1e-6 * rows.abs().max(), f"{node} is not the sum of its members (max gap {gap})"
print(df_merged[BA_COL].nunique(), "groups after adding interconnect/national aggregates")
print(df_merged[df_merged[BA_COL].isin(AGGREGATE_NODES)]
      .groupby(BA_COL)[TARGET].agg(["mean", "min", "max", "count"]))

In [ ]:
#train test split

# TRAIN_START / TRAIN_END / TEST_START / TEST_END come from config.py

df_merged = df_merged.sort_values(["balancing_authority_code_eia", "datetime_utc"])

train_mask = (df_merged["datetime_utc"] >= TRAIN_START) & (df_merged["datetime_utc"] <= TRAIN_END)
test_mask  = (df_merged["datetime_utc"] >= TEST_START) & (df_merged["datetime_utc"] <= TEST_END)

df_train = df_merged.loc[train_mask].copy()
df_test  = df_merged.loc[test_mask].copy()

# sub-period labels within test, for reporting no separate splits, just labels
# on the one continuous test set, so COVID-period performance can be isolated later
bounds = [
    df_test["datetime_utc"] < pd.Timestamp("2020-03-01"),
    df_test["datetime_utc"] < pd.Timestamp("2020-05-01"),
    df_test["datetime_utc"] < pd.Timestamp("2021-01-01"),
]
labels = ["pre_shock", "acute_covid", "recovery"]
df_test["subperiod"] = np.select(bounds, labels, default="stable")

print("Train:", df_train["datetime_utc"].min(), "to", df_train["datetime_utc"].max(), " rows:", len(df_train))
print("Test :", df_test["datetime_utc"].min(), "to", df_test["datetime_utc"].max(), " rows:", len(df_test))
print(df_test["subperiod"].value_counts())

In [ ]:
#GAM
from pygam import LinearGAM, s, f, te
import time

FEATURES = ["hour", "dayofweek", "dayofyear", "is_holiday",
            "load_24", "load_168", "avg_temp_K", "temp_s95", "temp_s99"]
TARGET = "demand_adjusted_mwh"

ba_data = {}  # ba_code: train/test data + fitted GAM + GAM predictions

for ba in tqdm(df_train["balancing_authority_code_eia"].unique(), desc="Fitting GAMs"):
    df_train_ba = df_train[df_train["balancing_authority_code_eia"] == ba].dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
    df_test_ba  = df_test[df_test["balancing_authority_code_eia"] == ba].dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)

    if len(df_train_ba) < 200 or len(df_test_ba) == 0:
        print(f"Skipping {ba}: insufficient data (train={len(df_train_ba)}, test={len(df_test_ba)})")
        continue

    X_train = df_train_ba[FEATURES].to_numpy(dtype=float)
    y_train = df_train_ba[TARGET].to_numpy(dtype=float)
    X_test  = df_test_ba[FEATURES].to_numpy(dtype=float)
    y_test  = df_test_ba[TARGET].to_numpy(dtype=float)

    gam = LinearGAM(
        s(0, n_splines=24, basis="cp") +   # hour, cyclic
        s(1, n_splines=7) +                 # dayofweek
        s(2, n_splines=10, basis="cp") +    # dayofyear, cyclic
        f(3) +                               # is_holiday
        s(4, n_splines=10) +                # load_24
        s(5, n_splines=10) +                # load_168
        s(6, n_splines=10) +                # avg_temp_K
        s(7, n_splines=10) +                # temp_s95
        s(8, n_splines=10) +                # temp_s99
        te(0, 6, n_splines=[8, 8])           # hour x avg_temp_K
    )
    _t0 = time.perf_counter()
    gam.gridsearch(X_train, y_train, lam=np.logspace(-3, 3, 11), progress=False)
    gam_s = time.perf_counter() - _t0

    ba_data[ba] = {
        "df_train": df_train_ba, "df_test": df_test_ba,
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_test,
        "gam": gam,
        "gam_pred_train": gam.predict(X_train),
        "gam_pred_test": gam.predict(X_test),
        "gam_s": gam_s,
    }
    print(f"{ba}: AIC={gam.statistics_['AIC']:.1f}  train_n={len(df_train_ba)}  test_n={len(df_test_ba)}")

print(f"\nFitted GAMs for {len(ba_data)} / {df_train['balancing_authority_code_eia'].nunique()} BAs")

In [ ]:
#XGB: stacked residual correction with rolling-window lambda optimization

import time
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from scipy.optimize import minimize_scalar

WINDOW = 2000
TUNE_WINDOW = 168
BATCH_SIZE = 24

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/timings", exist_ok=True)

final_results = {}
for ba, data in tqdm(ba_data.items(), desc="Stacking XGB"):
    ckpt_path = f"{RESULTS_DIR}/{ba}.parquet"
    if os.path.exists(ckpt_path):
        continue  

    df_train_ba, df_test_ba = data["df_train"], data["df_test"]
    X_train_feat = df_train_ba[FEATURES].astype(float)
    X_test_feat  = df_test_ba[FEATURES].astype(float)
    gam_pred_train, gam_pred_test = data["gam_pred_train"], data["gam_pred_test"]
    y_train_actual, y_test_actual = data["y_train"], data["y_test"]
    residuals_train = (y_train_actual - gam_pred_train).astype(float)
    X_train_xgb = X_train_feat.copy()
    X_train_xgb["gam_pred"] = gam_pred_train

    t0 = time.perf_counter()
    gs = GridSearchCV(
        estimator=xgb.XGBRegressor(objective="reg:squarederror", random_state=42),
        param_grid={"n_estimators": [100, 200], "max_depth": [3, 4]},
        cv=3, scoring="neg_mean_squared_error"
    )
    gs.fit(X_train_xgb, residuals_train)
    xgb_model = gs.best_estimator_
    t_grid = time.perf_counter() - t0

    X_test_xgb = X_test_feat.copy()
    X_test_xgb["gam_pred"] = gam_pred_test
    xgb_static_pred = xgb_model.predict(X_test_xgb) 

    t0 = time.perf_counter()
    current_X_history = X_train_xgb.copy()
    current_res_history = list(residuals_train)
    updated_preds, lambda_history, res_pred_history = [], [], []
    lambda_val = 1.0
    for i in range(0, len(X_test_xgb), BATCH_SIZE):
        X_batch = X_test_xgb.iloc[i:i + BATCH_SIZE]
        y_batch_actual = y_test_actual[i:i + BATCH_SIZE]
        gam_batch_pred = gam_pred_test[i:i + BATCH_SIZE]

        #predict next 24 hours
        res_pred = xgb_model.predict(X_batch)
        updated_preds.extend(gam_batch_pred + (lambda_val * res_pred))
        res_pred_history.extend(res_pred)
        lambda_history.extend([lambda_val] * len(X_batch))

        #apply residuals, roll window: drop oldest 24h, add newest 24h
        observed_residuals = y_batch_actual - gam_batch_pred
        current_res_history.extend(observed_residuals.tolist())
        current_X_history = pd.concat([current_X_history, X_batch]).iloc[-WINDOW:]
        current_res_history = current_res_history[-WINDOW:]

        #split window into 2 pieces, 1 refits 1 tunes lambda
        X_train_window = current_X_history.iloc[:-TUNE_WINDOW] #2000-168h
        res_train_window = current_res_history[:-TUNE_WINDOW]
        X_tune_window = current_X_history.iloc[-TUNE_WINDOW:] #most recent 168h
        res_tune_window = np.array(current_res_history[-TUNE_WINDOW:], dtype=float)
        xgb_model.fit(X_train_window, res_train_window) #refit XGB on last 1832h

        xgb_preds_tune = xgb_model.predict(X_tune_window) #evaluate on last 168h
        def objective_function(l):
            return np.mean((res_tune_window - (l * xgb_preds_tune)) ** 2)
        opt_result = minimize_scalar(objective_function, bounds=(0.0, 1.0), method="bounded")
        lambda_val = opt_result.x #tune lambda
    t_walk = time.perf_counter() - t0

    y_pred = np.array(updated_preds, dtype=float)
    y_true = y_test_actual[:len(y_pred)]
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))

    df_result_ba = df_test_ba.iloc[:len(y_pred)][
        ["datetime_utc", "balancing_authority_code_eia", "subperiod"]
    ].copy()
    df_result_ba["y_true"] = y_true
    df_result_ba["gam_pred"] = gam_pred_test[:len(y_pred)]
    df_result_ba["y_pred"] = y_pred
    df_result_ba["xgb_res_pred"] = np.asarray(res_pred_history, dtype=float)
    df_result_ba["xgb_static_pred"] = xgb_static_pred[:len(y_pred)]
    df_result_ba["lambda"] = lambda_history[:len(y_pred)]

    pd.DataFrame([{"node": ba, "gam_s": data.get("gam_s", np.nan), "xgb_grid_s": t_grid,
                   "walk_forward_s": t_walk, "n_train": len(df_train_ba), "n_test": len(df_test_ba)}]
                 ).to_csv(f"{RESULTS_DIR}/timings/{ba}.csv", index=False)
    df_result_ba.to_parquet(ckpt_path, index=False)  

    final_results[ba] = {"rmse": rmse, "mae": mae, "df_result": df_result_ba}
    print(f"{ba}: RMSE={rmse:.2f}  MAE={mae:.2f}  ({t_grid + t_walk:.0f} s)", flush=True)

print(f"\nFinished stacking XGB for {len(final_results)} / {len(ba_data)} BAs")